# Data Quality Toolkit — Complete Walkthrough

> **Version & safety posture.** This walkthrough was written against DQT v2.9.6; that functionality is included in v2.10.0.
> Sections are tiered: **Current local / implemented**, **v2.9.6 — added after v2.9.0**,
> and **PLANNED / FUTURE**. DQT runs **locally only** — no
> telemetry, no external API calls. It is **not** a SaaS, live service, deployment, or
> orchestration platform. Optional local AI, where documented elsewhere, is **default-off**;
> no model is bundled.

An end-to-end tour of DQT: profiling, quality assessment, preprocessing, export, drift /
monitoring, BI hand-off, and semantic catalogs. It extends the short
[01_quickstart.ipynb](01_quickstart.ipynb) into the full surface map.

> **Outputs stripped.** This notebook is distributed without cell outputs; run it locally to
> reproduce results.


## Audience & tier legend

Read top-to-bottom; the notebook layers by audience:

- **First-time user** — Setup through Preprocessing (sections B–E).
- **Technical evaluator / developer** — Export through Semantic catalogs (sections F–K).
- **Reviewer** — Boundary and Roadmap (sections N–O), readable as prose.

| Tier | Meaning | Marking |
|---|---|---|
| **Current local / implemented** | Implemented behavior in the source tree | default (no banner) |
| **v2.9.6 — added after v2.9.0** | Included in v2.10.0; current-star / proxy plus accepted UI/statistics/export posture | banner |
| **PLANNED / FUTURE** | Not implemented, not shipped | banner |


## Environment assumptions

- Python **3.12+**.
- Editable install: `pip install -e .`
- Optional extras, only where a section says so: `[stats]` (drift), `[ui]` (dashboard),
  `[powerbi]` (xlsx), `[duckdb]` (mirror), `[viz]` (plots). None are required for the core flow.
- **Run this notebook from the `examples/` directory** — every path below is relative.
- **No network calls.** Code cells write only to the git-ignored `dist/walkthrough/` directory.


In [ ]:
# Core public API (canonical seam: data_quality_toolkit.api; the root package re-exports it).
import pandas as pd

from data_quality_toolkit.api import (
    profile_csv,
    assess_csv,
    plan_csv,
    export_csv,
    compare_runs,
    generate_dim_time,
    build_powerbi_package,
    kpi_validate,
    kpi_emit,
    kpi_graph,
    get_builtin_quality_catalog,
    emit_builtin_quality_dax,
    validate_builtin_quality_semantic_contract,
)
from data_quality_toolkit.domain.profiling.charts import compute_univariate_chart_data

print("DQT core imports OK")


## Fixture strategy

This walkthrough uses only the small, fully synthetic fixtures that already ship in `examples/`:

| Fixture | Shape | Use |
|---|---|---|
| `demo/sample_orders.csv` | 170 × 14 | primary demo — profiling, assessment, export |
| `demo/issue_showcase/issue_demo.csv` | 5 × 4 | crafted dataset that triggers every issue type |
| `pipeline_gate/sample_pipeline_output.csv` | 10 × 5 | quality-gate example (score ≈ 0.66) |

> **Excluded: `demo/Uber_Data.csv`.** It is real-world aggregated data and is **not** confirmed
> license-cleared — it **requires license / PII review before any public use**. This notebook
> never reads it. The synthetic fixtures above are the preferred inputs.


In [ ]:
from pathlib import Path

# All paths are relative to the examples/ directory.
CSV_PATH = Path("demo/sample_orders.csv")                        # primary synthetic fixture (170x14)
ISSUE_CSV = Path("demo/issue_showcase/issue_demo.csv")           # crafted issue showcase (5x4)
PIPELINE_CSV = Path("pipeline_gate/sample_pipeline_output.csv")  # gate demo (score ~0.66)
KPI_CATALOG = Path("../config/kpi_catalog.yaml")                 # user-supplied KPI catalog (8 KPIs)

# Generated artifacts go here. dist/ is git-ignored, so nothing in here is committed.
OUT = Path("dist/walkthrough")
OUT.mkdir(parents=True, exist_ok=True)

for fixture in (CSV_PATH, ISSUE_CSV, PIPELINE_CSV):
    if not fixture.exists():
        raise FileNotFoundError(f"Fixture not found: {fixture}. Run from the examples/ directory.")

print("Fixtures located. Output dir:", OUT)


## 1. Profile — understand the dataset

`profile_csv` loads the CSV and returns structural statistics for the dataset and every column:
row count, dtypes, null counts, and cardinality. **No data is written to disk.**


In [ ]:
prof = profile_csv(CSV_PATH)
profile = prof["profile"]

print(f"run_id : {prof['run_id']}")
print(f"rows   : {profile['rows']:,}")
print(f"cols   : {profile['cols']}")
print(f"memory : {profile['memory_mb']:.3f} MB")
print()
print(f"{'column':<22}{'dtype':<14}{'null_pct':>9}{'unique':>9}")
print("-" * 54)
for col in profile["columns"]:
    null_pct = col["nulls"] / max(profile["rows"], 1)
    print(f"{col['name']:<22}{col['dtype']:<14}{null_pct:>8.1%}{str(col.get('unique', '?')):>9}")


## 2. EDA — column distributions

`compute_univariate_chart_data` returns binned distribution data for a column — the same
computation that the CLI renders as a terminal bar chart.

**CLI equivalent** (shown for reference; this cell is not executed by the notebook):

```bash
dqt chart demo/sample_orders.csv --column unit_price
```


In [ ]:
df = pd.read_csv(CSV_PATH)
chart = compute_univariate_chart_data(df, "unit_price")

print(f"column: {chart['column']}  |  type: {chart['type']}")
print()
max_count = max((count for _, count in chart["data"]), default=0) or 1
print(f"{'bin range':<28}{'count':>6}  distribution")
print("-" * 60)
for label, count in chart["data"]:
    bar = "#" * int(count / max_count * 30)
    print(f"{label:<28}{count:>6}  {bar}")


## Statistics boundary

The cells above are **descriptive** profiling, available in the core install. DQT also offers a
UI **Statistics Lab** tier for descriptive statistics, statistical tests, theoretical
Distributions, and Power & Sample Size planning. The inferential paths (normality checks,
Welch t-test / Mann-Whitney, ANOVA / Kruskal-Wallis, A/B comparison) are guarded by the
optional `[stats]` extra (scipy). Result tables use lightweight CSV downloads and optional
workspace saves where a Workspace Root is configured. Do not treat this notebook as running
the UI/statistics tier. When scipy is absent, those tests degrade gracefully with cautious
wording.


## 3. Quality assessment — detect issues and score

`assess_csv` runs quality rules over the profiled data and returns a 0–100 quality score plus a
typed issue list (severity, category, column). **No data is written to disk.**

Issue types you may see include: `missing` (nulls above threshold), `constant_column`,
`all_null_column`, `padded_column_name`, `placeholder_column_name`, `blank_column_name`, and
duplicate-column / duplicate-row flags.


In [ ]:
result = assess_csv(CSV_PATH)
assessment = result["assessment"]

print(f"quality score : {round(assessment['quality_score'] * 100, 1)}/100")
print(f"issues found  : {len(assessment['issues'])}")
print()
print(f"{'severity':<10}{'type':<30}column")
print("-" * 56)
for issue in assessment["issues"][:15]:
    column = issue.get("column") or "dataset-level"
    print(f"{issue.get('severity', '?'):<10}{issue.get('type', '?'):<30}{column}")


## 4. Quality gate

Use the score as a **pipeline gate**. The CLI exits `2` when the score is below the threshold,
so CI/ETL can halt and alert.

**CLI equivalent** (reference only — not executed):

```bash
# exit 0 = pass, 2 = below threshold, 1 = error
dqt assess demo/sample_orders.csv --fail-under 0.90
```

Defaults can also live in a project `dqt.yaml` (or the structured `config/v2_rules.yaml` rule
contract):

```yaml
# dqt.yaml
dataset:
  fail_under: 0.85
  score_field: quality_score
columns:
  customer_id: { required: true, critical: true }
  order_amount: { null_threshold: 0.05, weight: 2.0 }
```

> Note: `dqt.yaml` is read by the **CLI**, not by the Python API — in code, pass options as
> keyword arguments.


In [ ]:
# API gate: compare quality_score to a threshold and report PASS/FAIL.
# This notebook only reports; a real script would raise or set a process exit code here.
THRESHOLD = 0.90
score = assess_csv(CSV_PATH)["assessment"]["quality_score"]
verdict = "PASS" if score >= THRESHOLD else "FAIL"
print(f"score={score:.3f}  threshold={THRESHOLD:.2f}  ->  {verdict}")


## 5. Preprocessing plan

`plan_csv` inspects each column and returns actionable recommendations — null imputation,
encoding, outlier handling, scaling. **The source data is never mutated;** this is planning
output only, meant to guide ETL or feature engineering.


In [ ]:
plan = plan_csv(CSV_PATH)

print("dataset_id:", plan["dataset_id"])
print()
print(f"{'column':<22}{'dtype':<10}{'issues':<28}recommendations")
print("-" * 90)
for rec in plan["columns"]:
    print(f"{rec['column']:<22}{rec['dtype']:<10}{str(rec['issues']):<28}{rec['recommendations']}")


## 6. Export — star-schema quality artifacts

`export_csv` runs the full pipeline (profile → assess → star schema) and writes BI-ready
artifacts under an output directory **you choose**. Here that directory is the git-ignored
`dist/walkthrough/demo/` — nothing under `dist/` is committed.

Artifacts: `star/dim_dataset.csv`, `star/dim_column.csv`, `star/fact_profile_runs.csv`,
`star/fact_quality_metrics.csv`, `star/fact_issues.csv`, a per-run `quality_report.json`, an
append-only `quality_history.jsonl`, and an export-created `<output_dir>/dqt.db` run-history
database used by Quality History.


In [ ]:
export = export_csv(CSV_PATH, output_dir=str(OUT / "demo"))

print("export paths:")
for key, value in export["export_paths"].items():
    print(f"  {key}: {value}")

star_dir = OUT / "demo" / "star"
if star_dir.exists():
    print()
    print("star files:", sorted(p.name for p in star_dir.glob("*.csv")))


## 7. Compare runs

`compare_runs` reports how quality shifted between the latest two export runs for the same
dataset. With fewer than two runs it reports `not_enough_runs`, so the cell below exports a
second time first.


In [ ]:
# Export a second time so a prior run exists, then compare the latest two runs.
export_csv(CSV_PATH, output_dir=str(OUT / "demo"))
try:
    diff = compare_runs(CSV_PATH, output_dir=str(OUT / "demo"))
    print("score_delta:", diff.get("score_delta"))
except Exception as exc:
    print("compare unavailable:", exc)


## 8. Drift detection

Drift compares a baseline snapshot against a current snapshot — KS test for numeric columns,
chi-square for categorical. It needs the optional **`[stats]` extra (scipy)**.

**CLI equivalent** (reference only — not executed):

```bash
dqt drift baseline.csv current.csv --alpha 0.05 \
    --output drift_report.json --history drift_history.jsonl
```


In [ ]:
# Build two synthetic snapshots from the demo data (no external files, no network).
half = len(df) // 2
baseline_path = OUT / "baseline.csv"
current_path = OUT / "current.csv"
df.iloc[:half].to_csv(baseline_path, index=False)
df.iloc[half:].to_csv(current_path, index=False)

# Guarded: drift needs the optional [stats] extra. See docs/cli.md for the full flag set.
try:
    from data_quality_toolkit.api import detect_drift
    drift = detect_drift(baseline_path, current_path)
    print("drift result keys:", list(drift.keys()))
except Exception as exc:
    print("drift detection skipped:", exc)
    print('Enable it with: pip install "data-quality-toolkit[stats]"')


## 9. Drift history & monitoring

Append-only drift history (JSONL) can be promoted into a **SQLite** monitoring store, then
queried for trends and per-column metrics (PSI / Jensen-Shannon / Wasserstein). Threshold gates
exit `2` on breach, and a **dependency-free static HTML dashboard** (no server, no JavaScript)
can be rendered for CI artifacts.

> DQT is **not** a live monitoring service. These are local, file-based analytics.

**CLI equivalents** (reference only — not executed):

```bash
dqt drift-history import drift_history.jsonl --db monitoring.db
dqt drift-history trend   --db monitoring.db --fail-on-drift-rate 0.30
dqt drift-history columns --db monitoring.db --fail-on-psi 0.20
dqt drift-history dashboard --db monitoring.db --output drift-dashboard.html --include-plots
```


In [ ]:
# Monitoring helpers (guarded discovery only — no fragile calls, no network).
# Exact signatures live in docs/api.md / docs/cli.md; run drift with --history to produce input.
try:
    import data_quality_toolkit.api as dqt_api
    helpers = [
        name
        for name in (
            "import_drift_history_sqlite",
            "summarize_drift_trends_sqlite",
            "evaluate_drift_rate_threshold",
            "evaluate_psi_threshold",
            "read_drift_columns_sqlite",
        )
        if hasattr(dqt_api, name)
    ]
    print("available monitoring helpers:", helpers)
    print("See docs/api.md for exact signatures.")
except Exception as exc:
    print("monitoring helpers unavailable:", exc)


## Drift notifications (dry-run only)

DQT can build a drift-threshold webhook payload. It is **dry-run by default** — it prints the
JSON payload and sends nothing. A real POST requires `--send` **and** `DQT_ALLOW_NETWORK=true`,
and is SSRF-guarded (the destination host must resolve to a public address).

> **This notebook never sends a notification.** The URL below is a non-routable placeholder.

**CLI equivalent** (reference only — not executed):

```bash
dqt drift-history notify --db monitoring.db --webhook-url https://hooks.example.com/dqt
```


## 10. User-defined KPI catalog (current / implemented)

Define your **own** business KPIs in YAML, validate them, and emit Power BI DAX + a TMSL model.
The repo ships a worked example at `config/kpi_catalog.yaml` (8 KPIs over the quality star
schema). These commands operate on **catalogs you supply** — they are separate from DQT's own
built-in catalog (next section).


In [ ]:
# Guarded: reads ../config/kpi_catalog.yaml and writes under OUT.
try:
    validation = kpi_validate(str(KPI_CATALOG))
    print("kpi catalog status:", validation.get("status"), "| kpis:", validation.get("kpis"))

    kpi_emit(
        str(KPI_CATALOG),
        dax_out=str(OUT / "semantics" / "measures.dax"),
        tmsl_out=str(OUT / "semantics" / "model.tmsl.json"),
    )
    graph = kpi_graph(
        str(KPI_CATALOG),
        out=str(OUT / "semantics" / "kpi_graph"),
        graph_format="mermaid",
    )
    print("dependency graph:", graph.get("graph"))
except Exception as exc:
    print("kpi catalog example skipped:", exc)


In [ ]:
# Time dimension + Power BI package, both written under the git-ignored OUT directory.
dim = generate_dim_time("2022-01-01", "2026-12-31", output_dir=str(OUT / "time"))
print("dim_time rows:", dim.get("rows"), "| path:", dim.get("path"))

star_dir = OUT / "demo" / "star"
try:
    if star_dir.exists():
        package = build_powerbi_package(str(star_dir), str(OUT / "powerbi_package"))
        print("package dir:", package.get("package_dir"))
        print("files:", package.get("files"))
    else:
        print(f"Run the export cell first to create {star_dir}.")
except Exception as exc:
    print("power bi package skipped:", exc)


## BI hand-off — what DQT emits vs what you do in Power BI

`build_powerbi_package` assembles a zero-config package under the output directory: the quality
`star/` CSVs, `time/dim_time.csv`, `relationships.json`, a single `BaseFolder` Power Query
parameter in `parameters.json`, a `model.pbit` (or a `model.pbit.README` placeholder when no
template is bundled), and a `README.txt`.

`dim_time.csv` carries `time_id` (YYYYMMDD), `date`, calendar fields, ISO week fields, a weekend
flag, and — when `--fiscal` is set — `fiscal_year` / `fiscal_quarter`. In Power BI you load it,
**Mark as date table** on `date`, and relate facts on `time_id`.

> The star schema describes **data quality** (profiling runs and quality metrics), not business
> sales — that is the point: a BI team can monitor dataset health over time. DQT does **not**
> generate incremental-refresh policies, RLS roles, or service settings, and it does **not**
> deliver a full Power BI semantic model — you author measures and secure the model in Power BI.


## 11. Built-in quality semantic catalog

> **v2.9.6 — added after v2.9.0; included in v2.10.0. Current-star / proxy scope only.
> Do not claim full canonical BI semantic model completion.** The Quality Score here is the
> labeled `quality_score_proxy` (title *"Quality Score (proxy)"*), **not** the true assessment
> `quality_score`. Intentionally **held / absent**: a true Quality Score, drift / PSI measures,
> a duplicate-rate measure, a failed-quality-gate count, and a Last Run Date.

DQT ships a small built-in **current-star** quality semantic catalog: the quality measures it can
express over its **own** star-schema / Power BI package outputs. This is separate from the
user-supplied catalog above.


In [ ]:
# Built-in current-star / proxy catalog. emit_builtin_quality_dax() with no path returns the DAX
# text and writes nothing. Contract validation runs only if a package was built earlier.
catalog = get_builtin_quality_catalog()
print("built-in catalog version:", catalog.get("version"), "| measures:", catalog.get("measure_count"))

dax = emit_builtin_quality_dax()  # no output_path -> returns text, no write
print("dax measure_count:", dax.get("measure_count"), "| output_path:", dax.get("output_path"))

package_dir = OUT / "powerbi_package"
try:
    if package_dir.exists():
        check = validate_builtin_quality_semantic_contract(str(package_dir))
        print("contract valid:", check.get("valid"), "| mode:", check.get("input_mode"))
    else:
        print(f"Build the Power BI package first to validate it ({package_dir}).")
except Exception as exc:
    print("contract validation skipped:", exc)


## 12. Pipeline runner / workflow scaffold

`dqt run` (and the `create_elt_pipeline` / `create_manifest` API) is a **declarative scaffold /
dry-run** workflow: it records step selections and reads existing session artifacts.

> **DQT run is not an ETL engine or orchestration platform.** It does not transform data, load
> data into targets, or connect to warehouses — pipeline execution and scheduling remain your
> responsibility.

An illustrative HTTP-service workflow exists in a local-only `n8n/` directory; it is **not
shipped** and is not relied on as committed user-facing material.

**CLI equivalent** (reference only — not executed):

```bash
dqt run --run-id RUN_ID --sessions-root dist/sessions/
```


## 13. UI walkthrough (read-only here)

DQT has an optional Streamlit dashboard (`pip install data-quality-toolkit[ui]`) organised as a
16-page / 16-step product journey — Start / Load Dataset, Data Overview, EDA Explorer,
Statistics Lab, Quality Score, Preprocess Studio, Pipeline Runner, Drift Monitoring, Quality
History, Artifact Center, Export, KPI Catalog, Dim Time, Manifest Viewer, Settings /
Governance, and Help / About.

Statistics Lab includes Distributions, Power & Sample Size, lightweight CSV downloads, and
opt-in workspace saves. Export creates/uses `<output_dir>/dqt.db`, which Quality History reads
for trends and run comparison.

> **This notebook does not run Streamlit and does not test the UI.** No screenshots are included.
> For the full UI walkthrough, see **[dashboard/README.md](dashboard/README.md)**.

**Launchers** (reference only — not executed):

```bash
dqt dashboard               # launch the Streamlit dashboard
dqt ui --db monitoring.db   # preferred launcher; seeds the Drift Monitoring page
```


## Boundary & do-not-claim

To keep this walkthrough honest:

- **Version posture:** v2.9.6 functionality is included in v2.10.0.
- **Built-in quality catalog** is **current-star / proxy** scope (`quality_score_proxy`) — not a
  full BI semantic model.
- **Do not claim** DQT is a SaaS, live service, deployment, orchestration platform, runtime
  platform, microservice, or production service; it is local and file-based.
- **Do not claim** the full canonical Power BI semantic model is complete, that optional local AI
  runs by default, or that any model is bundled.
- **`demo/Uber_Data.csv`** is not confirmed license-cleared and is excluded from this notebook;
  it requires license / PII review before any public use.
- This notebook has **not** been executed or output-stripping-validated here, and is **not**
  published or snapshot-ready — those steps belong to later gates.


## Roadmap / future

> **PLANNED / FUTURE — not implemented, not shipped, do not rely on.** The full canonical DQT BI
> semantic model — including a true Quality Score, Drift Rate, Max / Average PSI, Duplicate Rate,
> Failed-Quality-Gate Count, and Last Run Date — and the n8n HTTP `/v1/export` workflow are
> planned for later releases. They are not present in the toolkit today.

Other planned / tracked items: consolidation of duplicate example fixtures and a shared
formatting / threshold-validation kernel shared across CLI and UI.


## Cleanup & reproducibility

- All generated artifacts live under `dist/walkthrough/` (git-ignored). Remove it any time:
  `rm -rf dist/walkthrough` (PowerShell: `Remove-Item -Recurse -Force dist/walkthrough`).
- **Before committing this notebook: strip all outputs and clear execution counts.** There is no
  `nbstripout` hook configured, so this is a manual step (for example
  `jupyter nbconvert --clear-output --inplace complete_dqt_walkthrough.ipynb`).
- Use **relative paths only**; never embed absolute machine paths.
- Do **not** commit anything generated under `dist/`.
- The flow is deterministic given the same inputs and DQT version.
